<a href="https://colab.research.google.com/github/NagaTejaSriCore/Buliding-Game-Using-Crew/blob/main/Buliding__game_crew_sec3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U crewai

In [ ]:
!pip install "crewai[google-genai]"

In [ ]:
!pip install "crewai[tools]"

In [ ]:
from crewai import Agent,LLM,Task,Crew,Process
from google.colab import userdata

In [ ]:
from crewai_tools import SerperDevTool

In [ ]:
serper_tool=SerperDevTool(api_key=userdata.get("serper_api_key"))

#**Creative Game Designer Agent**
```
   role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
   
   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
```

In [ ]:
from crewai import Agent,LLM
from google.colab import userdata

In [ ]:
llm=LLM(
    model="gemini-2.5-flash",
    api_key=userdata.get("google_api")
)

In [ ]:
designer=Agent(
    role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",

   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
    llm=llm,
    verbose=True,
    max_iter=1
)

#**Senior Python Game Developer Agent**

```
role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
```

In [ ]:
developer=Agent(
    role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
    llm=llm,
    verbose=True,
    max_iter=1,
    tools=[serper_tool]
)

#**QA Engineer & Code Reviewer Agent**

    role="QA Engineer & Code Reviewer",

    goal="Test, review, and improve the code for bugs, playability, and completeness",
    
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",


In [ ]:

QA_engineer=Agent(role="QA Engineer & Code Reviewer",

goal="Test, review, and improve the code for bugs, playability, and completeness",

backstory=
    "You are a meticulous QA engineer and code reviewer."
    "You carefully check:"
    "- Does the code run without errors?"
    "- Does it implement ALL the designed features?"
    "- Is it fun/playable? Any obvious balance issues?"
    "- Code style, variable names, comments"
    "Suggest fixes or small improvements and output the FINAL improved code.",
                  llm=llm,
                  max_iter=1,
                  verbose=True)

### **Game Designing Task**

    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",

In [ ]:
task_design=Task(
    description=
    "Take the user's game idea: {game_idea}"
    "1. Clarify and expand it into a fun, simple 2D game"
    "2. Describe: objective, controls, entities, win/lose"
    "3. Keep scope small (one level, basic mechanics)"
    "Output format:"
    "## Game Design Document"
    "- Title: ..."
    "- Genre: ..."
    "- Objective: ..."
    "- Controls: ..."
    "- Entities: ..."
    "- Mechanics: ...",
    expected_output="A clear Markdown Game design Document",
    agent=designer
)

### **Coding Task**

    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main",
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",

In [ ]:
task_code=Task(
    description=
    "Using the game design from the previous task"
    "Write a COMPLETE, standalone Python script using Pygame that implements the game."
    "- Include import pygame, sys, random (if needed)"
    "- Full game loop, init, events, update, draw"
    "- Make it runnable with python game.py"
    "- Add simple comments"
    "- The main game loop must be exposed in the python code, it should not be inside any function like main"
    "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",
    expected_output="A Complete runnable Pygame Python script",
    agent=developer,
    context=[task_design]
)

###**Review Task**

    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    

In [ ]:
task_review=Task(
    description=
    "Review the Python code from the previous task."
    "1. Check for syntax/runtime errors"
    "2. Verify it matches the design document"
    "3. Test mentally: does it have init, loop, quit handling, drawing?"
    "4. Suggest fixes/improvements if needed"
    "5. Output the FINAL, improved, ready-to-run code"
    "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final polished, runnable Pygame Python script and instructions on how to play the game",
    agent=QA_engineer,
    context=[task_design,task_code]
)

In [ ]:
game_crew=Crew(
    agents=[designer,developer,QA_engineer],
    tasks=[task_design,task_code,task_review],
    process=Process.sequential,
    verbose=True
)

In [ ]:
game_idea="A fun endless runner where a character jumps over obstacles"
result = await game_crew.kickoff_async(
    inputs={"game_idea": game_idea}
)

print(result)

In [ ]:
%%writefile main.py

import pygame
import random

# Initialize Pygame
pygame.init()

# Screen dimensions
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 400

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255)
GREY = (100, 100, 100)

# Game constants
GROUND_HEIGHT = 50
PLAYER_WIDTH = 20
PLAYER_HEIGHT = 40
PLAYER_X = 80

GRAVITY = 0.8
JUMP_STRENGTH = -15

OBSTACLE_MIN_WIDTH = 20
OBSTACLE_MAX_WIDTH = 50
OBSTACLE_MIN_HEIGHT = 30
OBSTACLE_MAX_HEIGHT = 80
OBSTACLE_START_X = SCREEN_WIDTH + 50

INITIAL_GAME_SPEED = 5
GAME_SPEED_INCREMENT = 0.005

OBSTACLE_SPAWN_INTERVAL = 1500
MIN_OBSTACLE_SPAWN_INTERVAL = 700

SCORE_INCREMENT_PER_FRAME = 0.1

# Set up display
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Pixel Leaper")

# Clock
clock = pygame.time.Clock()

# Fonts
font = pygame.font.Font(None, 36)
game_over_font = pygame.font.Font(None, 64)


# Obstacle class
class Obstacle:
    def __init__(self, x, y, width, height, speed):
        self.rect = pygame.Rect(x, y, width, height)
        self.speed = speed

    def update(self):
        self.rect.x -= self.speed

    def draw(self, screen):
        pygame.draw.rect(screen, BLUE, self.rect)


# Reset game function
def reset_game():
    global player_y
    global player_y_velocity
    global is_grounded
    global obstacles
    global score
    global game_active
    global game_speed
    global last_obstacle_spawn_time

    player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
    player_y_velocity = 0
    is_grounded = True
    obstacles = []
    score = 0
    game_active = True
    game_speed = INITIAL_GAME_SPEED
    last_obstacle_spawn_time = pygame.time.get_ticks()


# Initial game state
player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
player_y_velocity = 0
is_grounded = True

obstacles = []

score = 0

game_active = True

game_speed = INITIAL_GAME_SPEED

last_obstacle_spawn_time = pygame.time.get_ticks()


# Main game loop
running = True

while running:

    # Events
    for event in pygame.event.get():

        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.KEYDOWN:

            if event.key == pygame.K_SPACE:

                if game_active:

                    if is_grounded:
                        player_y_velocity = JUMP_STRENGTH
                        is_grounded = False

                else:
                    reset_game()

    # Game updates
    if game_active:

        # Gravity
        player_y_velocity += GRAVITY
        player_y += player_y_velocity

        # Ground collision
        if player_y >= SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT:
            player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
            player_y_velocity = 0
            is_grounded = True

        # Increase difficulty
        game_speed += GAME_SPEED_INCREMENT
        score += SCORE_INCREMENT_PER_FRAME

        # Spawn obstacles
        current_time = pygame.time.get_ticks()

        current_spawn_interval = max(
            MIN_OBSTACLE_SPAWN_INTERVAL,
            OBSTACLE_SPAWN_INTERVAL - (int(score) // 100) * 50
        )

        if current_time - last_obstacle_spawn_time > current_spawn_interval:

            obstacle_width = random.randint(
                OBSTACLE_MIN_WIDTH,
                OBSTACLE_MAX_WIDTH
            )

            obstacle_height = random.randint(
                OBSTACLE_MIN_HEIGHT,
                OBSTACLE_MAX_HEIGHT
            )

            obstacle_y = (
                SCREEN_HEIGHT
                - GROUND_HEIGHT
                - obstacle_height
            )

            new_obstacle = Obstacle(
                OBSTACLE_START_X,
                obstacle_y,
                obstacle_width,
                obstacle_height,
                game_speed
            )

            obstacles.append(new_obstacle)

            last_obstacle_spawn_time = current_time

        # Update obstacles
        for obstacle in list(obstacles):

            obstacle.update()

            if obstacle.rect.right < 0:
                obstacles.remove(obstacle)

        # Collision detection
        player_rect = pygame.Rect(
            PLAYER_X,
            player_y,
            PLAYER_WIDTH,
            PLAYER_HEIGHT
        )

        for obstacle in obstacles:

            if player_rect.colliderect(obstacle.rect):
                game_active = False

        # Draw background
        screen.fill(WHITE)

        # Draw ground
        pygame.draw.rect(
            screen,
            GREY,
            (0, SCREEN_HEIGHT - GROUND_HEIGHT,
             SCREEN_WIDTH, GROUND_HEIGHT)
        )

        # Draw player
        pygame.draw.rect(screen, RED, player_rect)

        # Draw obstacles
        for obstacle in obstacles:
            obstacle.draw(screen)

        # Draw score
        score_text = font.render(
            f"Score: {int(score)}",
            True,
            BLACK
        )

        screen.blit(score_text, (10, 10))

    else:

        # Game Over screen
        screen.fill(WHITE)

        game_over_text = game_over_font.render(
            "GAME OVER",
            True,
            BLACK
        )

        game_over_rect = game_over_text.get_rect(
            center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 30)
        )

        screen.blit(game_over_text, game_over_rect)

        final_score_text = font.render(
            f"Final Score: {int(score)}",
            True,
            BLACK
        )

        final_score_rect = final_score_text.get_rect(
            center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 20)
        )

        screen.blit(final_score_text, final_score_rect)

        restart_text = font.render(
            "Press SPACE to Restart",
            True,
            BLACK
        )

        restart_rect = restart_text.get_rect(
            center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 60)
        )

        screen.blit(restart_text, restart_rect)

    # Update display
    pygame.display.flip()

    # FPS
    clock.tick(60)

pygame.quit()

# **Do not edit any of the below code**

In [ ]:
!pip install pygbag pyngrok -q

# **How to get pyngrok API Key?**
- Go to https://dashboard.ngrok.com/get-started/your-authtoken
- Copy your authtoken
- Run this cell with YOUR token:

In [ ]:
ngrok_auth_token=userdata.get("NGROK_AUTH_TOKEN")

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', 'main.py'],
    capture_output=True, text=True
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Start HTTP server
    print("\n🚀 Starting server...")
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000', '--directory', '/content/build/web'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Step 3: Create ngrok tunnel
    try:
        print("🌐 Creating public URL...")
        public_url = ngrok.connect(8000)

        print("\n" + "="*60)
        print("🎮 YOUR GAME IS READY!")
        print("="*60)
        print(f"\n🔗 Click here to play: {public_url}")
        print("\n📝 How to play:")
        print("   • Press SPACE or Click to jump")
        print("="*60)

    except Exception as e:
        print(f"\n❌ Error creating tunnel: {e}")
        print("\n💡 Make sure you ran Cell 3 with a valid ngrok token")